In [44]:
from numba import cuda
import numpy as np
import time
import math

In [45]:
@cuda.jit
def matrix_multiplication(A,B,C):
  row, col = cuda.grid(2)

  if(row < C.shape[0] and col < C.shape[1]):
    temp = 0

    for k in range(A.shape[1]):
      temp += A[row, k] * B[k, col]

    C[row,col] = temp

In [46]:
N = 256
A = np.random.randint(0,10,(N,N)).astype(np.float32)
B = np.random.randint(0,10,(N,N)).astype(np.float32)
C_gpu = np.zeros((N,N),dtype=np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(C_gpu)

In [47]:
threads_per_block = (16,16)

blocks_per_cell_x = math.ceil(N/threads_per_block[0])
blocks_per_cell_y = math.ceil(N/threads_per_block[1])

blocks_per_cell = (blocks_per_cell_x,blocks_per_cell_y)

In [48]:
start_gpu = time.perf_counter()
matrix_multiplication[blocks_per_cell,threads_per_block](d_A,d_B,d_C)
cuda.synchronize()
end_gpu = time.perf_counter()
gpu_time = (end_gpu - start_gpu) * 1000
C_gpu = d_C.copy_to_host()

In [49]:
start_cpu = time.perf_counter()

C_cpu = np.zeros((A.shape[0],B.shape[1]))

for i in range(A.shape[0]):
  for j in range(B.shape[0]):
    for k in range(A.shape[1]):
      C_cpu[i,j] += A[i,k] * B[k,j]

end_cpu = time.perf_counter()
cpu_time = (end_cpu - start_cpu) * 1000

In [50]:
print("Matrix A:")
print(A[:5,:5])

print("Matrix B:")
print(B[:5,:5])

print("Matrix C (GPU):")
print(C_gpu[:5,:5])

print("Matrix C (CPU):")
print(C_cpu[:5,:5])

print("GPU time: ",gpu_time,"ms")
print("CPU time: ",cpu_time,"ms")

Matrix A:
[[7. 8. 0. 7. 5.]
 [7. 2. 1. 1. 5.]
 [1. 4. 0. 8. 7.]
 [2. 8. 9. 8. 8.]
 [2. 4. 8. 8. 8.]]
Matrix B:
[[5. 5. 0. 0. 4.]
 [6. 8. 2. 1. 8.]
 [6. 6. 8. 6. 2.]
 [4. 2. 5. 6. 7.]
 [6. 7. 7. 5. 6.]]
Matrix C (GPU):
[[5273. 5575. 5906. 5295. 5484.]
 [5318. 5310. 5774. 5174. 5079.]
 [4953. 5015. 5239. 4911. 5088.]
 [5177. 5229. 5435. 5206. 5035.]
 [5067. 5284. 5444. 5228. 4880.]]
Matrix C (CPU):
[[5273. 5575. 5906. 5295. 5484.]
 [5318. 5310. 5774. 5174. 5079.]
 [4953. 5015. 5239. 4911. 5088.]
 [5177. 5229. 5435. 5206. 5035.]
 [5067. 5284. 5444. 5228. 4880.]]
GPU time:  131.16438799988828 ms
CPU time:  14458.418789000007 ms
